<a href="https://colab.research.google.com/github/Yurnero2706/DataAlgo-UT/blob/main/DataAlgo2026_R06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ構造とアルゴリズム 第6週課題 (2026/05/27 ver.A)

---


★下記３要素を**必ず直接書き換えてから**提出すること．

* 学籍番号： 202318032
* 氏名： Nguyen Cong Nguyen
* Colabアカウント： ncnguyencva@gmail.com

**学生同士で教えた・教わった・グループワークをした場合，必ず下記の当該行を直接書き換えて記述すること．**  
申告があった場合は，論述などで内容が似ていたり，プログラムが似ていても減点はしない．（コピペレベルの同一文などは処罰対象）  
**教えた側は加点対象となる．**

**[教えた側]**  
教えた相手：＜氏名＞　＜学籍番号＞  
（何名いてもよい；教えた相手の内容が浅くなってないか確認すること）

**[教わった側]**  
教わった相手：＜氏名＞　＜学籍番号＞  
（何名いてもよい；必ず教えた側に自分の名前を書いてもらうこと）

**[グループワーク]**  
一緒に行った相手：＜氏名＞　＜学籍番号＞  
（何名いてもよいが，必ずお互いの名前を全員記すこと）



---
# 必須課題6A Dijkstraの解法での最短経路の表示の仕組み

DataAlgo_UT(008)_ShortestPath1N.ipynb で示した shortest_dijkstra プログラムにおいて，それぞれの頂点への最小経路値を実現する経路を**表示する**仕組みをプログラムに即して説明せよ．  
**経路を求める仕組みの説明を求めているのではないことに注意すること**．  
（経路を求める過程を説明してもよいがそれだけでは不十分である）  
このとき，構造体CostFromのメンバ from に格納されている値について言及すること．  
適宜プログラムを参照すること．

* 本授業内容（講義資料参照）のプログラムに即した説明であること．
* （ChatGPT等に相談すると授業内容と異なることが多いので注意すること．これは本授業の課題すべてに共通）
* 図解において頂点選択に同等の選択肢がある場合は頂点番号の小さいものから選ぶこと．

(1) プログラム中の構造体CostFromのメンバ変数まで言及しつつテキストセル(1)とコードセル(1)で説明せよ．
(2) グラフ6において，開始頂点を4として図解で説明せよ．YouTube Shortの形（２週目の課題参照）でテキストセル(2)にURLを示すこと．（手書きで紙に書くこと．肉声で解説しながら書くこと．本人確認も兼ねてるので，ペンタブや合成音声などは全て認めません）  
例 https://youtube.com/shorts/T2YnvHBectU  

The `shortest_dijkstra` program saves **one** piece of routing information: which vertex came right before it on the best path found so far. That single piece of information is the `from` member of the `CostFrom` struct.

The struct looks like this:

```c
typedef struct {
    int cost;
    int from;
} CostFrom;

CostFrom vertexinfo[N];
```

Each vertex `v` has its own `vertexinfo[v]`. The meaning of `vertexinfo[v].from` is *"to reach v along the shortest path, the last edge comes from vertex vertexinfo[v].from"*. So we never store an entire path anywhere — we only store one predecessor per vertex. Across all vertices, the `from` values form a tree rooted at `StartVertex`, and any shortest path from `StartVertex` to `v` can be recovered by walking up this tree starting from `v`.

The value of `from` is set every time the relaxation step finds a shorter route:

```c
if (vertexinfo[min_vertex].cost + edge[min_vertex][v] < vertexinfo[v].cost) {
    vertexinfo[v].cost = vertexinfo[min_vertex].cost + edge[min_vertex][v];
    vertexinfo[v].from = min_vertex;   // <-- record the predecessor
}
```

So `from` always points back along the best path known at this moment. If a better path is found later, both `cost` and `from` get overwritten together.

**The display mechanism itself** is the backward-chasing loop at the end of `find_shortest_dijkstra()`:

```c
for (s = v; s != StartVertex; s = vertexinfo[s].from)
    printf("%2d <- ", s);
printf("%2d", StartVertex);
```

Reading from left to right, this loop says:

1. Start at the destination vertex `v`. Print it.
2. Jump to its predecessor: `s = vertexinfo[s].from`. Print it.
3. Keep jumping to predecessors and printing, until we reach `StartVertex` itself.
4. Finally print `StartVertex` separately, outside the loop, to close the path.

The loop terminates correctly because at initialization we set `vertexinfo[StartVertex].from = StartVertex`, so once `s` becomes the start, the loop condition `s != StartVertex` is false. The path is printed in **reverse order** (destination first, source last), separated by `<-`. This direction matches how the predecessors were stored.

For vertices that are unreachable, `vertexinfo[v].from` stays at `-1` (the initial value for non-start vertices), and the cost stays at `NC`. The display code prints `"No path"` for these vertices instead of running the backward loop.

**Summary.** The display works by chasing `from` pointers backward. Each vertex only contributed one entry, but because `from` was correctly updated together with `cost` every time a shorter path was found, walking these pointers in reverse automatically gives the whole shortest path. No path list ever exists in memory — it is reconstructed at print time.

In [ ]:
typedef struct {
    int cost;   // best known cost from StartVertex to this vertex
    int from;   // the previous vertex on that best-known path
} CostFrom;

CostFrom vertexinfo[N];

// (1) Initialization: StartVertex's "from" points to itself; others to -1
for (v = 0; v < N; v++) {
    if (v == StartVertex) {
        vertexinfo[v].cost = 0;
        vertexinfo[v].from = StartVertex;   // <-- self-loop sentinel for the start
    } else {
        vertexinfo[v].cost = NC;
        vertexinfo[v].from = -1;
    }
}

for (v = 0; v < N; v++) {
    if (checkelement(v) >= 0) {
        if (vertexinfo[min_vertex].cost + edge[min_vertex][v] < vertexinfo[v].cost) {
            vertexinfo[v].cost = vertexinfo[min_vertex].cost + edge[min_vertex][v];
            vertexinfo[v].from = min_vertex;   // <-- record the predecessor
        }
    }
}

// (3) The display: walk the "from" chain backward from destination to StartVertex
for (s = v; s != StartVertex; s = vertexinfo[s].from)
    printf("%2d <- ", s);
printf("%2d", StartVertex);


テキストセル(2) URL

---
# 必須課題6B Dijkstraプログラムの計算量


DataAlgo_UT(008)_ShortestPath1N.ipynb で示した shortest_diskstra プログラムの時間計算量と空間計算量を，プログラムの構造を参照しつつ議論せよ．

* 授業内で言及していた方針に従ってまとめること
* （ChatGPT等に相談すると授業内での言及と異なる説明をしてくることが多いので鵜呑みにしないこと．これは本授業の課題すべてに共通）  
* 説明に必要なプログラム参照については，以下のコードセルに必要部分**のみ**を張り付けること．  
* 文章による説明は以下のテキストセルを利用せよ．



Let $N$ be the number of vertices and $\text{edge}[N][N]$ the adjacency matrix.

**Time complexity:**

The main `while` loop runs once per element removed from the set $U$. Since $U$ starts with $N$ vertices and each iteration removes exactly one, the loop runs exactly $N$ times.

Inside each iteration, the cost is dominated by three blocks (see code excerpt below):

- `obtainelementlist(restS, N)` — scans the whole `setU` array of size $N$ to copy out the remaining elements. Cost: $O(N)$.
- The "find min" loop — scans `restS[]` (size $\leq N$) for the smallest cost. Cost: $O(N)$.
- The relaxation loop — runs $N$ times, calling `checkelement(v)` and comparing costs. Each call is $O(1)$. Cost: $O(N)$.

Total work per iteration: $O(N)$. Multiply by $N$ iterations: $N \times O(N) = O(N^2)$.

After the loop, the display section also runs in $O(N^2)$ — for each of the $N$ vertices, the backward `from`-walk follows at most $N$ predecessors (since a simple path can't have more than $N$ vertices). So display does not change the asymptotic time.

**Total: $O(N^2)$.**

(With a binary heap as the priority queue, Dijkstra can be reduced to $O((N+E) \log N)$, but the array-based set used here gives a linear scan for finding the min, so we stay at $O(N^2)$.)

**Space complexity**

Auxiliary memory:

- `vertexinfo[N]`: array of structs with 2 `int`s each. Cost $O(N)$.
- `setU[N]` (inside `SetLib_J`) is $O(N)$.
- `restS[N]`: temporary list of remaining elements. $O(N)$.

So auxiliary space is $O(N)$. The adjacency matrix `edge[N][N]` is $O(N^2)$ but counts as input, not working memory.

**Total: $O(N)$.**

In [ ]:
while (restelements > 0) {                          // (*) outer loop: runs N times in total

    obtainelementlist(restS, N);                    // O(N): scans setU[] of size N

    // Find the vertex with the minimum tentative cost
    min_vertex = restS[0];
    min_cost   = vertexinfo[min_vertex].cost;
    for (i = 1; i < restelements; i++) {            // O(N): scans restS[]
        v = restS[i];
        if (vertexinfo[v].cost < min_cost) {
            min_vertex = v;
            min_cost   = vertexinfo[v].cost;
        }
    }

    deleteelement(min_vertex);                      // O(1)
    restelements--;

    for (v = 0; v < N; v++) {                       // O(N): one full row of edge[][]
        if (checkelement(v) >= 0) {                 // O(1)
            if (vertexinfo[min_vertex].cost + edge[min_vertex][v] < vertexinfo[v].cost) {
                vertexinfo[v].cost = vertexinfo[min_vertex].cost + edge[min_vertex][v];
                vertexinfo[v].from = min_vertex;
            }
        }
    }
}


---
# 必須課題6C Floydの解法での最短経路の表示の仕組み

DataAlgo_UT(009)_ShortestPathNN.ipynb で示した shortest_floyd のCプログラムにおいて，全ての頂点対の最小経路を表示する部分には再帰呼出関数を用いている．  
これについて，本プログラムが最小経路を**表示していく仕組み**を説明せよ．  
**経路を求める仕組みの説明を求めているのではないことに注意すること**．  
（経路を求める過程を説明してもよいがそれだけでは不十分である）  
特に，via[][]が格納している値について言及しながら説明すること．  
適宜プログラムを参照すること．


* 本授業内容（講義資料参照）のプログラムに即した説明であること．
* （ChatGPT等に相談すると授業内容と異なることが多いので注意すること．これは本授業の課題すべてに共通）
* 図解において頂点選択に同等の選択肢がある場合は頂点番号の小さいものから選ぶこと．

(1) プログラム中の変数viaの格納している値の意味に言及しつつテキストセル(1)とコードセル(1)で説明せよ．
(2) グラフ6において，配列pathinfo[][]が得られた後，頂点0から頂点5に至る経路を求める過程を図解で説明せよ．開始前にpathinfo[][]の配列を書いておくこと（印刷でもよい）．YouTube Shortの形（２週目の課題参照）でテキストセル(2)にURLを示すこと．（手書きで紙に書くこと．肉声で解説しながら書くこと．本人確認も兼ねてるので，ペンタブや合成音声などは全て認めません）  
例 https://youtube.com/shorts/T2YnvHBectU  

The display in `shortest_floyd` uses a recursive function `outputpath()`. The key piece of information is the field `via` inside the struct stored at `pathinfo[s][d]`.

**The recursive display function:**

```c
void outputpath(int SrcNode, int DstNode){
    if (SrcNode == pathinfo[SrcNode][DstNode].via) {
        printf("%2d ", SrcNode);                         // base case
    } else {
        outputpath(SrcNode, pathinfo[SrcNode][DstNode].via);
        outputpath(pathinfo[SrcNode][DstNode].via, DstNode);
    }
}
```

1. **Base case:** `via[s][d] == s`. The path s→d is a direct edge, no intermediate. So we just print `s`.
2. **Recursive case:** `via[s][d] = k` for some `k` different from `s`. Then the optimal path goes s→...→k→...→d, where each "..." part is itself an optimal subpath. We solve those two subpaths recursively: first the left half (s to k), then the right half (k to d).

The recursion always shrinks the problem: each recursive call refers to a path that was built **earlier** in Floyd's loop (it used a smaller index `k`), so the recursion is guaranteed to bottom out at base cases — direct edges.

`via[s][d]` stores either the source itself (meaning "direct edge") or some intermediate vertex on the optimal path. The recursive function expands each `via` into two subpaths, and the recursion terminates at direct edges where it prints the source. The print order traces the path from start to end.

In [ ]:
// Relevant excerpts from shortest-floyd_J.c

typedef struct {
    int cost;   // best known cost from s to d
    int via;    // an intermediate vertex on the s-to-d shortest path,
                //   or s itself if the path is a direct edge
} CostVia;

CostVia pathinfo[N][N];

// (1) Initialization: assume direct-edge only.
//     via is set to the source (sentinel meaning "no intermediate yet").
for (SrcNode = 0; SrcNode < N; SrcNode++) {
    for (DstNode = 0; DstNode < N; DstNode++) {
        pathinfo[SrcNode][DstNode].cost = edge[SrcNode][DstNode];
        pathinfo[SrcNode][DstNode].via  = SrcNode;   // <-- "no intermediate"
    }
}

// (2) Main loop: whenever going through MidNode helps, record it in via.
for (MidNode = 0; MidNode < N; MidNode++) {
    for (SrcNode = 0; SrcNode < N; SrcNode++) {
        for (DstNode = 0; DstNode < N; DstNode++) {
            NewCost = pathinfo[SrcNode][MidNode].cost
                    + pathinfo[MidNode][DstNode].cost;
            if (NewCost < pathinfo[SrcNode][DstNode].cost) {
                pathinfo[SrcNode][DstNode].cost = NewCost;
                pathinfo[SrcNode][DstNode].via  = MidNode;   // <-- record midpoint
            }
        }
    }
}

// (3) The display: recursively split the path at its via point.
void outputpath(int SrcNode, int DstNode){
    if (SrcNode == pathinfo[SrcNode][DstNode].via) {
        printf("%2d ", SrcNode);                                 // base: direct edge, print source
    } else {
        outputpath(SrcNode, pathinfo[SrcNode][DstNode].via);     // left half: s -> ... -> via
        outputpath(pathinfo[SrcNode][DstNode].via, DstNode);     // right half: via -> ... -> d
    }
}
// Caller appends the final DstNode after outputpath returns:
//     outputpath(SrcNode, DstNode);
//     printf("%2d", DstNode);


テキストセル(2) URL

---
# 必須課題6D Floyd プログラムの計算量

DataAlgo_UT(009)_ShortestPathNN.ipynb で示した shortest_floyd プログラムの時間計算量と空間計算量を，プログラムの構造を参照しつつ議論せよ．

* 授業内で言及していた方針に従ってまとめること
* （ChatGPT等に相談すると授業内での言及と異なる説明をしてくることが多いので鵜呑みにしないこと．これは本授業の課題すべてに共通）  
* 説明に必要なプログラム参照については，以下のコードセルに必要部分**のみ**を張り付けること．  
* 文章による説明は以下のテキストセルを利用せよ．



Let $N$ be the number of vertices.

**Time complexity**

The body of `find_shortest_floyd()` has two main parts:

1. **Initialization**: a doubly nested loop over all $(s, d)$ pairs, with $O(1)$ work inside. Cost: $O(N^2)$.

2. **The main Floyd loop**: a *triply nested* loop:

```c
for (MidNode = 0; MidNode < N; MidNode++) {       // N iterations
    for (SrcNode = 0; SrcNode < N; SrcNode++) {   // N iterations
        for (DstNode = 0; DstNode < N; DstNode++) { // N iterations
            // O(1): one addition, one comparison, maybe two writes
        }
    }
}
```

**Total time: $O(N^3)$.**

This is *not* better than running Dijkstra $N$ times ($N \cdot O(N^2) = O(N^3)$ as well), but Floyd's constant factors are tiny because the inner body is just an add-compare-store with no set operations.

**Space complexity**

Auxiliary memory:

- `pathinfo[N][N]` — 2D array of structs (2 ints each). Cost: $O(N^2)$.
- Scalar variables (`MidNode`, `SrcNode`, `DstNode`, `NewCost`, etc.). $O(1)$.
- Recursion stack for `outputpath()` during display — depth at most $N$ (the path length). $O(N)$.

So auxiliary space is dominated by `pathinfo[N][N]` at $O(N^2)$. The adjacency matrix `edge[N][N]` is also $O(N^2)$ but is input.

**Total auxiliary space: $O(N^2)$.**

In [ ]:
// The triple-nested loop that decides the time complexity of shortest_floyd

for (MidNode = 0; MidNode < N; MidNode++) {              // O(N)
    for (SrcNode = 0; SrcNode < N; SrcNode++) {          // O(N)
        for (DstNode = 0; DstNode < N; DstNode++) {      // O(N)
            NewCost = pathinfo[SrcNode][MidNode].cost
                    + pathinfo[MidNode][DstNode].cost;   // O(1)
            if (NewCost < pathinfo[SrcNode][DstNode].cost) {
                pathinfo[SrcNode][DstNode].cost = NewCost;
                pathinfo[SrcNode][DstNode].via  = MidNode;
            }
        }
    }
}
// Total: N * N * N * O(1) = O(N^3) time.
// Memory: pathinfo[N][N] of structs = O(N^2) auxiliary space.


---
# 必須課題6E Dijkstraのプログラムを再利用したN対N最短経路推定プログラム

DataAlgo_UT(008)_ShortestPath1N.ipynb で示した shortest_diskstra プログラムを改良して，N対Nの最短経路問題を解くプログラムを実現せよ．  
プログラム内で開始頂点を1からNまで変更しながら，dijkstraのアルゴリズムを実行するという方針である．  
改良したプログラムについて，時間計算量と空間計算量を議論せよ．



In [8]:
%%writefile SetLib_J.h
// Set management
// 2020/05/26 kameda[at]ccs.tsukuba.ac.jp

// １つの集合のみ扱う．
// その集合の要素は非負の整数とする．

// 集合の初期化
// 引数：確保する要素数
// 返値：成功...確保した整数配列のポインタ, 失敗...NULL
int *initset(int );

// 集合への追加
// 指定された（非負）整数を集合に追加する．
// 引数：集合に加える要素番号
// 返値：成功 ... 指定された整数, 失敗 ... 負値
int addelement(int );

// 集合から要素vを削除
// 指定された整数を集合から削除する．
// 引数：集合から削除する要素番号
// 返値：成功 ... 指定された整数, 失敗 ... 負値
int deleteelement(int );

// 要素vが集合内にあるかどうかを調査
// 引数：調査対象とする要素番号
// 返値：成功 ... 指定された要素番号, 失敗 ... 負値
int checkelement(int );

// 集合の要素一覧の入手
// 引数：要素を納める配列へのポインタ、配列の要素数
// 返値：成功時には要素数、失敗時は負値
int obtainelementlist(int *, int );

// 集合の状態表示
// 引数：なし
// 返値：成功時には現在の要素数、失敗時は負値
int showset(void);



Writing SetLib_J.h


In [9]:
%%writefile SetLib_J.c
// Set management
// 2021/05/26 kameda[at]ccs.tsukuba.ac.jp
#include <stdio.h> // printf()
#include <stdlib.h> // calloc()
#include "SetLib_J.h" // プロトタイプ宣言の整合確認

int *setU = NULL;  // 頂点集合 U を表す配列
int setsize = 0; // 集合 U の最大要素数

// 集合の初期化
// 引数：確保する要素数
// 返値：成功...確保した整数配列のポインタ, 失敗...NULL
int *initset(int n) {
    if (setU != NULL) {
		printf("【失敗】%d 要素分がすでに確保済です。\n", setsize);
        return NULL;
    }
	setU = calloc(n, sizeof(*setU));
	if (setU == NULL) {
		printf("【失敗】 %d要素分の集合用配列を確保できませんでした．\n", n);
        return NULL;
	}
    setsize = n;
	return setU;
}

// 集合への追加
// 指定された（非負）整数を集合に追加する．
// 引数：集合に加える要素番号
// 返値：成功 ... 指定された整数, 失敗 ... 負値
int addelement(int v) {
    if (setU == NULL) {
        printf("【失敗】 集合が用意できていません．\n");
        return -1;
    } else if (v < 0 || v >= setsize) {
        printf("【失敗】 指定された要素番号 %d は有効範囲 (0 - %d) 外です\n", v, setsize - 1);
        return -2;
    }
    setU[v] = 1;
    return v;
}

// 集合から要素vを削除
// 指定された整数を集合から削除する．
// 引数：集合から削除する要素番号
// 返値：成功 ... 指定された整数, 失敗 ... 負値
int deleteelement(int v) {
    if (setU == NULL) {
        printf("【失敗】 集合が用意できていません．\n");
        return -1;
    } else if (v < 0 || v >= setsize) {
        printf("【失敗】 指定された要素番号 %d は有効範囲 (0 - %d) 外です\n", v, setsize - 1);
        return -2;
    }
    setU[v] = 0;
    return v;
}

// 要素vが集合内にあるかどうかを調査
// 引数：調査対象とする要素番号
// 返値：成功 ... 指定された要素番号, 失敗 ... 負値
int checkelement(int v) {
    if (setU == NULL) {
        printf("【失敗】 集合が用意できていません．\n");
        return -1;
    } else if (v < 0 || v >= setsize) {
        printf("【失敗】 指定された要素番号 %d は有効範囲 (0 - %d) 外です\n", v, setsize - 1);
        return -2;
    }

    if (setU[v] == 0)
        return -3;
    return v;
};

// 集合の要素一覧の入手
// 引数：要素を納める配列へのポインタ、配列の要素数
// 返値：成功時には要素数、失敗時は負値
int obtainelementlist(int *array, int n) {
    int t; // 集合中に現在存在している要素数
    int i, j;

    if (setU == NULL) {
        printf("【失敗】 集合が用意できていません．\n");
        return -1;
    }

    for (i = 0, t = 0; i < setsize; i++) {
        if (setU[i] != 0)
            t++;
    }

    if (n < t) {
        printf("【失敗】配列の要素数 %d が集合の現在の要素数 %d より小さいため入りきりません．\n", n, t);
        return -2;
    }

    for (i = 0, j = 0; i < setsize; i++) {
        if (setU[i] != 0) {
            array[j] = i;
            j++;
        }
    }

    return t;
}

// 集合の状態表示
// 引数：なし
// 返値：成功時には現在の要素数、失敗時は負値
int showset(void) {
    int t; // 集合中に現在存在している要素数
    int i;

    if (setU == NULL) {
        printf("【失敗】 集合が用意できていません．\n");
        return -1;
    }

    printf("全要素数 %d\n", setsize);
    printf("  集合 : ");
    for (i = 0, t = 0; i < setsize; i++) {
        if (setU[i] != 0) {
            printf("%d ", i);
            t++;
        }
    }
    printf("\n");
    printf("  現在の要素数 %d\n", t);

    return t;
}



Writing SetLib_J.c


In [13]:
%%writefile graph6.h
// 8 nodes, undirected, no loop, positive weight.
// NC means no edges.
// NC will be treated as "inifinity" on searching the shortest path.
#define N 8
#define NC 9999 // this big value means both no path and infinity
int edge[N][N] = {
//     0   1   2   3   4   5   6   7
	{  0,100, NC, NC,145,200, NC, NC}, // 0
	{100,  0, 23, NC, NC, 80, NC, 70}, // 1
	{ NC, 23,  0, 21, NC, 45, 37, NC}, // 2
	{ NC, NC, 21,  0, NC, NC, 18,  5}, // 3
	{145, NC, NC, NC,  0, 20, NC, NC}, // 4
	{200, 80, 45, NC, 20,  0,  2, NC}, // 5
	{ NC, NC, 37, 18, NC,  2,  0,  1}, // 6
	{ NC, 70, NC,  5, NC, NC,  1,  0}  // 7
};

Writing graph6.h


In [14]:
%%writefile Report6E.c
#include <stdio.h>
#include <stdlib.h>
#include "SetLib_J.h"
#include "graph6.h"   // gives N, NC, edge[N][N]

typedef struct {
    int cost;
    int from;
} CostFrom;

CostFrom result[N][N];   // result[s][.] is what Dijkstra found when starting from s

// Run Dijkstra starting from StartVertex; fill in result[StartVertex][.]
void dijkstra_from(int StartVertex){
    int v, i;
    int restelements;
    int restS[N];
    int min_vertex, min_cost;

    // Refill the set U with all vertices (addelement is idempotent here)
    for (v = 0; v < N; v++) {
        addelement(v);
    }
    restelements = N;

    // Initialize result[StartVertex][.]
    for (v = 0; v < N; v++) {
        if (v == StartVertex) {
            result[StartVertex][v].cost = 0;
            result[StartVertex][v].from = StartVertex;
        } else {
            result[StartVertex][v].cost = NC;
            result[StartVertex][v].from = -1;
        }
    }

    // Standard Dijkstra main loop
    while (restelements > 0) {
        obtainelementlist(restS, N);

        min_vertex = restS[0];
        min_cost   = result[StartVertex][min_vertex].cost;
        for (i = 1; i < restelements; i++) {
            v = restS[i];
            if (result[StartVertex][v].cost < min_cost) {
                min_vertex = v;
                min_cost   = result[StartVertex][v].cost;
            }
        }

        deleteelement(min_vertex);
        restelements--;

        for (v = 0; v < N; v++) {
            if (checkelement(v) >= 0) {
                if (result[StartVertex][min_vertex].cost + edge[min_vertex][v]
                    < result[StartVertex][v].cost) {
                    result[StartVertex][v].cost =
                        result[StartVertex][min_vertex].cost + edge[min_vertex][v];
                    result[StartVertex][v].from = min_vertex;
                }
            }
        }
    }
}

int main(int argc, char *argv[]){
    int s, d, x;

    // Allocate the set once.
    initset(N);

    // Run Dijkstra from every vertex.
    for (s = 0; s < N; s++) {
        dijkstra_from(s);
    }

    // Print all-pairs shortest paths
    printf("=== All-pairs shortest paths ===\n");
    for (s = 0; s < N; s++) {
        for (d = 0; d < N; d++) {
            printf("%d -> %d : ", s, d);
            if (s == d) {
                printf("cost 0 (self)\n");
            } else if (result[s][d].cost == NC) {
                printf("No path\n");
            } else {
                printf("cost %3d : ", result[s][d].cost);
                for (x = d; x != s; x = result[s][x].from)
                    printf("%d <- ", x);
                printf("%d\n", s);
            }
        }
        printf("\n");
    }
    return 0;
}


Overwriting Report6E.c


In [15]:
!echo "コンパイルと実行コマンドをここに（提出時に出力をつけ)
!gcc -Wall -c SetLib_J.c
!gcc -Wall -c Report6E.c
!gcc -Wall -o Report6E Report6E.o SetLib_J.o
!./Report6E


/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file
=== All-pairs shortest paths ===
0 -> 0 : cost 0 (self)
0 -> 1 : cost 100 : 1 <- 0
0 -> 2 : cost 123 : 2 <- 1 <- 0
0 -> 3 : cost 144 : 3 <- 2 <- 1 <- 0
0 -> 4 : cost 145 : 4 <- 0
0 -> 5 : cost 152 : 5 <- 6 <- 7 <- 3 <- 2 <- 1 <- 0
0 -> 6 : cost 150 : 6 <- 7 <- 3 <- 2 <- 1 <- 0
0 -> 7 : cost 149 : 7 <- 3 <- 2 <- 1 <- 0

1 -> 0 : cost 100 : 0 <- 1
1 -> 1 : cost 0 (self)
1 -> 2 : cost  23 : 2 <- 1
1 -> 3 : cost  44 : 3 <- 2 <- 1
1 -> 4 : cost  72 : 4 <- 5 <- 6 <- 7 <- 3 <- 2 <- 1
1 -> 5 : cost  52 : 5 <- 6 <- 7 <- 3 <- 2 <- 1
1 -> 6 : cost  50 : 6 <- 7 <- 3 <- 2 <- 1
1 -> 7 : cost  49 : 7 <- 3 <- 2 <- 1

2 -> 0 : cost 123 : 0 <- 1 <- 2
2 -> 1 : cost  23 : 1 <- 2
2 -> 2 : cost 0 (self)
2 -> 3 : cost  21 : 3 <- 2
2 -> 4 : cost  49 : 4 <- 5 <- 6 <- 7 <- 3 <- 2
2 -> 5 : cost  29 : 5 <- 6 <- 7 <- 3 <- 2
2 -> 6 : cost  27 : 6 <- 7 <- 3 <- 2
2 -> 7 : cost  26 : 7 <- 3 

**Time complexity — $O(N^3)$.**

`dijkstra_from(s)` is the same Dijkstra algorithm from Task 6B, so each call is $O(N^2)$. We call it once for every starting vertex, so:

$${N} \;\times\; O(N^2) \;=\; O(N^3).$$

The final print step is $N^2$ vertex pairs, with at most $N$ steps to walk back through the `from` chain for each pair. So the display is also $O(N^3)$ in the worst case. This matches but does not exceed the algorithm itself.

**Total: $O(N^3)$.** Same asymptotic order as `shortest_floyd`, but with larger constant factors because of the set operations inside SetLib.

**Space complexity is $O(N^2)$.**

- `result[N][N]` — array of structs storing the full N-to-N answer. $O(N^2)$.
- `setU[N]` from SetLib — $O(N)$.
- `restS[N]` — $O(N)$.
- Scalars — $O(1)$.

The biggest auxiliary array is `result[N][N]`, so total auxiliary space is $O(N^2)$. This is the same as Floyd's — we now store the full all-pairs table, so we cannot avoid the $O(N^2)$ space.


---
---
# 発展課題6X 十分大きな数NC

DataAlgo_UT(008)_ShortestPath1N.ipynb で示した shortest_diskstra プログラムや DataAlgo_UT(009)_ShortestPathNN.ipynb で示した shortest_floyd プログラムにおいて，
あるグラフが与えられた時，暫定経路値の一時的最大値と最大重み辺が分かれば，NCを決めることができる．これを正確に求めることは難しいが，アルゴリズムの趣旨を考えれば，これは正確な値である必要はない．そこで，グラフの隣接行列が与えられたとして，NCの近似値として役に立つ数字を 時間計算量 O(N^2) で求める簡単な方法を考えてみよ．


（議論はここに記述）

---
# 課題提出法

筑波大学工学システム学類３年生向け．  
FG24711 / FG34711．  

必須課題は全て実施すること．
発展課題はしなくともよいが，A+取得には発展課題を（全課題提出を通して）1つ以上実施していることが必要条件である．







# 出典

筑波大学工学システム学類  
データ構造とアルゴリズム  
担当：亀田能成  
  
---
2026/05/27 ver.A  
